In [1]:
!pip install datasets==2.19.0 transformers seqeval evaluate accelerate -q

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

PyTorch: 2.10.0+cu128
CUDA: True


In [3]:
raw_dataset = load_dataset("conll2003", trust_remote_code=True)
example = raw_dataset['train'][0]
print("Tokens :", example['tokens'])
print("POS    :", example['pos_tags'])
print("Chunk  :", example['chunk_tags'])

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokens : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
POS    : [22, 42, 16, 21, 35, 37, 16, 21, 7]
Chunk  : [11, 21, 11, 12, 21, 22, 11, 12, 0]


In [4]:
pos_label_names   = raw_dataset['train'].features['pos_tags'].feature.names
chunk_label_names = raw_dataset['train'].features['chunk_tags'].feature.names
print(f"POS labels ({len(pos_label_names)})   :", pos_label_names)
print(f"Chunk labels ({len(chunk_label_names)}) :", chunk_label_names)

POS labels (47)   : ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk labels (23) : ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [5]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("Tokenizer loaded:", type(tokenizer).__name__)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: BertTokenizer


In [6]:
def tokenize_and_align_labels(examples, label_column):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

In [7]:
pos_tokenized = raw_dataset.map(
    lambda x: tokenize_and_align_labels(x, 'pos_tags'),
    batched=True,
    remove_columns=raw_dataset['train'].column_names
)
chunk_tokenized = raw_dataset.map(
    lambda x: tokenize_and_align_labels(x, 'chunk_tags'),
    batched=True,
    remove_columns=raw_dataset['train'].column_names
)
print(pos_tokenized)
print(chunk_tokenized)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})


In [8]:
pos_id2label = {i: l for i, l in enumerate(pos_label_names)}
pos_label2id = {l: i for i, l in enumerate(pos_label_names)}
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_names),
    id2label=pos_id2label,
    label2id=pos_label2id
)
print("POS model ready, labels:", len(pos_label_names))

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


POS model ready, labels: 47


In [9]:
chunk_id2label = {i: l for i, l in enumerate(chunk_label_names)}
chunk_label2id = {l: i for i, l in enumerate(chunk_label_names)}
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_names),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)
print("Chunk model ready, labels:", len(chunk_label_names))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Chunk model ready, labels: 23


In [10]:
seqeval = evaluate.load("seqeval")
def make_compute_metrics(label_names):
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions = np.argmax(logits, axis=-1)
        true_labels, true_preds = [], []
        for pred_row, label_row in zip(predictions, labels):
            tl, tp = [], []
            for p, l in zip(pred_row, label_row):
                if l == -100:
                    continue
                tl.append(label_names[l])
                tp.append(label_names[p])
            true_labels.append(tl)
            true_preds.append(tp)
        r = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            "precision": round(r["overall_precision"], 4),
            "recall":    round(r["overall_recall"],    4),
            "f1":        round(r["overall_f1"],        4),
            "accuracy":  round(r["overall_accuracy"],  4),
        }
    return compute_metrics

In [14]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
pos_trainer = Trainer(
    model=pos_model,
    args=TrainingArguments(
        output_dir="./results_pos",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
    ),
    train_dataset=pos_tokenized['train'],
    eval_dataset=pos_tokenized['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_label_names),
)
pos_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.752358,0.262684,0.908500,0.910200,0.909300,0.936500
2,0.204865,0.233282,0.917900,0.915500,0.916700,0.942800
3,0.163772,0.223856,0.920100,0.919400,0.919800,0.944800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2634, training_loss=0.30293381893010235, metrics={'train_runtime': 266.9395, 'train_samples_per_second': 157.8, 'train_steps_per_second': 9.867, 'total_flos': 510472720030266.0, 'train_loss': 0.30293381893010235, 'epoch': 3.0})

In [16]:
chunk_trainer = Trainer(
    model=chunk_model,
    args=TrainingArguments(
        output_dir="./results_chunk",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
    ),
    train_dataset=chunk_tokenized['train'],
    eval_dataset=chunk_tokenized['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_label_names),
)
chunk_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.458068,0.205985,0.908300,0.899300,0.903800,0.948900
2,0.167816,0.180943,0.914600,0.910100,0.912400,0.953600
3,0.134518,0.175096,0.915300,0.911900,0.913600,0.954400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2634, training_loss=0.2172304586469901, metrics={'train_runtime': 276.3432, 'train_samples_per_second': 152.43, 'train_steps_per_second': 9.532, 'total_flos': 510251380802730.0, 'train_loss': 0.2172304586469901, 'epoch': 3.0})

In [17]:
pos_results   = pos_trainer.evaluate(eval_dataset=pos_tokenized['test'])
chunk_results = chunk_trainer.evaluate(eval_dataset=chunk_tokenized['test'])
print("POS Tagging:")
print(f"  Precision: {pos_results['eval_precision']}  Recall: {pos_results['eval_recall']}  F1: {pos_results['eval_f1']}")
print("\nChunking:")
print(f"  Precision: {chunk_results['eval_precision']}  Recall: {chunk_results['eval_recall']}  F1: {chunk_results['eval_f1']}")

POS Tagging:
  Precision: 0.9125  Recall: 0.9077  F1: 0.9101

Chunking:
  Precision: 0.9057  Recall: 0.897  F1: 0.9013


In [18]:
def predict_tags(sentence, model, label_names):
    model.eval()
    words = sentence.split()
    inputs = tokenizer(words, is_split_into_words=True, return_tensors='pt').to(device)
    model.to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    pred_ids = logits.argmax(dim=-1).tolist()
    word_ids = inputs.word_ids()
    results, seen = [], set()
    for idx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        results.append((words[wid], label_names[pred_ids[idx]]))
    return results

In [19]:
sentences = [
    "John works at Google in California .",
    "The quick brown fox jumps over the lazy dog .",
    "Hugging Face released a new transformer model yesterday .",
]
for sent in sentences:
    pos_out   = predict_tags(sent, pos_model,   pos_label_names)
    chunk_out = predict_tags(sent, chunk_model, chunk_label_names)
    print(f"\nInput: {sent}")
    print(f"{'Token':<20} {'POS':<12} {'Chunk'}")
    print("-" * 45)
    for (tok, pos), (_, chk) in zip(pos_out, chunk_out):
        print(f"{tok:<20} {pos:<12} {chk}")


Input: John works at Google in California .
Token                POS          Chunk
---------------------------------------------
John                 NNP          B-NP
works                VBZ          B-VP
at                   IN           B-PP
Google               NNP          B-NP
in                   IN           B-PP
California           NNP          B-NP
.                    .            O

Input: The quick brown fox jumps over the lazy dog .
Token                POS          Chunk
---------------------------------------------
The                  DT           B-NP
quick                JJ           I-NP
brown                NNP          I-NP
fox                  NNP          I-NP
jumps                VBZ          B-VP
over                 IN           B-PP
the                  DT           B-NP
lazy                 JJ           I-NP
dog                  NN           I-NP
.                    .            O

Input: Hugging Face released a new transformer model yesterday .
Token 

In [20]:
comparison = pd.DataFrame({
    'Aspect'      : ['Label Count', 'F1 Score', 'Precision', 'Recall', 'Label Format', 'Difficulty'],
    'POS Tagging' : [len(pos_label_names),   pos_results['eval_f1'],
                     pos_results['eval_precision'],  pos_results['eval_recall'],
                     'Flat (NN, VB…)', 'Easier'],
    'Chunking'    : [len(chunk_label_names), chunk_results['eval_f1'],
                     chunk_results['eval_precision'], chunk_results['eval_recall'],
                     'IOB2 (B-NP, I-NP…)', 'Medium'],
})
print(comparison.to_string(index=False))

      Aspect    POS Tagging           Chunking
 Label Count             47                 23
    F1 Score         0.9101             0.9013
   Precision         0.9125             0.9057
      Recall         0.9077              0.897
Label Format Flat (NN, VB…) IOB2 (B-NP, I-NP…)
  Difficulty         Easier             Medium
